# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de flota

Este notebook adapta el **Notebook 1 original de AndinaLog** a `andinalog_flota`. Mantiene la lógica: **leer origen → diagnosticar → marcar cuarentena → generar reporte → exportar**, sin corregir los datos en esta etapa.

El origen en Drive es `Giad/Dataset/andinalog_flota.csv`.


## 1 · Configuración y origen

Está preparado para Google Colab. Monta Google Drive y busca automáticamente el archivo dentro de `MyDrive/Giad/Dataset/`.

**No necesitas cambiar la ruta cada vez.** Si el archivo tiene otro nombre, el código también muestra los CSV/XLSX disponibles en esa carpeta.


In [2]:
from pathlib import Path
import hashlib
import os
import sys
import tempfile
import pandas as pd


# ============================================================
# CONFIGURACIÓN
# ============================================================

# "auto"  -> detecta automáticamente Colab o entorno local
# "local" -> fuerza ejecución local
# "drive" -> fuerza ejecución en Google Colab + Drive
ENTORNO = "auto"

# Ruta raíz del proyecto cuando se ejecuta en Google Colab
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"

# Carpeta Bronze común para el proyecto
CARPETA_DATASETS = "AndinaLog_03B_Bronce"

# Dataset utilizado por este notebook
NOMBRE_ARCHIVO = "andinalog_flota.csv"

VERSION_DIAGNOSTICO = "GIAD-M3-S4-FLOTA-diagnostico-v1"


# ============================================================
# DETECCIÓN DE LA RAÍZ DEL PROYECTO EN LOCAL
# ============================================================

def encontrar_raiz_local():
    """
    Busca la raíz del proyecto desde la carpeta actual
    y sus carpetas padre.
    """

    for carpeta in [Path.cwd(), *Path.cwd().parents]:

        if (
            (carpeta / "datasets" / CARPETA_DATASETS).is_dir()
            and (carpeta / "S4").is_dir()
        ):
            return carpeta

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Ejecuta el notebook dentro de practicasNotebookColab."
    )


# ============================================================
# CONFIGURACIÓN AUTOMÁTICA DE RUTAS
# ============================================================

def configurar_rutas(entorno, ruta_drive):

    # --------------------------------------------------------
    # Detectar automáticamente el entorno
    # --------------------------------------------------------

    if entorno == "auto":
        entorno = (
            "drive"
            if "google.colab" in sys.modules
            else "local"
        )

    # --------------------------------------------------------
    # Google Colab + Google Drive
    # --------------------------------------------------------

    if entorno == "drive":

        from google.colab import drive

        drive.mount("/content/drive")

        raiz = Path(ruta_drive)

    # --------------------------------------------------------
    # Ejecución local
    # --------------------------------------------------------

    elif entorno == "local":

        raiz = encontrar_raiz_local()

    else:

        raise ValueError(
            "ENTORNO debe ser 'auto', 'local' o 'drive'"
        )

    # --------------------------------------------------------
    # Ruta del dataset Bronze
    # --------------------------------------------------------

    bronze = (
        raiz
        / "datasets"
        / CARPETA_DATASETS
        / NOMBRE_ARCHIVO
    )

    # --------------------------------------------------------
    # Directorio de salidas del notebook
    # --------------------------------------------------------

    salidas = (
        raiz
        / "S4"
        / "andinalog_flota"
        / "notebook1"
        / "salidas"
    )

    return entorno, raiz, bronze, salidas


# ============================================================
# OBTENER RUTAS
# ============================================================

ENTORNO_DETECTADO, RUTA_RAIZ, RUTA_BRONZE, DIRECTORIO_SALIDAS = (
    configurar_rutas(
        ENTORNO,
        RUTA_PROYECTO_DRIVE
    )
)


# ============================================================
# VERIFICAR DATASET
# ============================================================

if not RUTA_BRONZE.is_file():

    raise FileNotFoundError(
        f"No se encontró el CSV Bronze: {RUTA_BRONZE}"
    )


# ============================================================
# MOSTRAR CONFIGURACIÓN
# ============================================================

print("Entorno:", ENTORNO_DETECTADO)
print("Raíz:", RUTA_RAIZ)
print("Bronze:", RUTA_BRONZE)
print("Salidas:", DIRECTORIO_SALIDAS)


Entorno: local
Raíz: c:\Users\remrodri\Github\practicasNotebookColab
Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_flota.csv
Salidas: c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_flota\notebook1\salidas


## 2 · Carga y contrato

La fuente real que estamos usando contiene **32 filas y 4 columnas**:

- `camion_id`
- `centro_distribucion_base`
- `capacidad_kg`
- `tipo_camion`

Se conservan los valores originales. Las conversiones usadas para validar capacidad son temporales y no se exportan como datos corregidos.


In [3]:
COLUMNAS_ORIGINALES = [
    'camion_id',
    'centro_distribucion_base',
    'capacidad_kg',
    'tipo_camion'
]

def cargar_origen(ruta):
    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()

    if ruta.suffix.lower() == '.csv':
        df = pd.read_csv(
            ruta,
            dtype='string',
            encoding='utf-8-sig',
            keep_default_na=False
        )
    else:

        df = pd.read_excel(
            ruta,
            dtype='string',
            keep_default_na=False
        )

    df.columns = [str(c).strip() for c in df.columns]
    return df, huella

df_bronze, HASH_BRONZE = cargar_origen(RUTA_BRONZE)

faltantes = [c for c in COLUMNAS_ORIGINALES if c not in df_bronze.columns]
if faltantes:
    raise ValueError(
        f'Faltan columnas requeridas: {faltantes}. '
        f'Columnas encontradas: {list(df_bronze.columns)}'
    )

df_bronze = df_bronze[COLUMNAS_ORIGINALES].copy()

print(f'Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas')
print('SHA-256:', HASH_BRONZE)
display(df_bronze.head())


Bronze: 32 filas × 4 columnas
SHA-256: 5c02eb8ef591c7f98846b8f547f41383467b41e7768bddb4031bb43d1b812bec


,camion_id,centro_distribucion_base,capacidad_kg,tipo_camion
0,CAM-01,Cochabamba,12000,Refrigerado
1,CAM-02,Oruro,2000,Seco
2,CAM-03,Tarija,5000,Refrigerado
3,CAM-04,Santa Cruz,12000,Seco
4,CAM-05,Oruro,5000,Refrigerado


## 3 · Catálogo y reglas de diagnóstico

El diagnóstico **no corrige** ningún valor.

Se consideran problemas:
- identificador faltante;
- identificador con formato inconsistente;
- identificador duplicado;
- centro de distribución faltante;
- capacidad faltante, no numérica o menor/igual a cero;
- tipo de camión faltante o no reconocido.

Los tipos válidos para este dataset son `Refrigerado` y `Seco`.


In [4]:
CATALOGO_PROBLEMAS = pd.DataFrame([
    ('camion_id', 'FALTANTE', 'Identificador vacío'),
    ('camion_id', 'FORMATO_INCONSISTENTE', 'El identificador no sigue el patrón CAM-##'),
    ('camion_id', 'DUPLICADO', 'Identificador repetido; se marca la aparición posterior'),
    ('centro_distribucion_base', 'FALTANTE', 'Centro de distribución vacío'),
    ('capacidad_kg', 'FALTANTE', 'Capacidad vacía'),
    ('capacidad_kg', 'NO_NUMERICA', 'Valor no convertible a número'),
    ('capacidad_kg', 'FUERA_RANGO', 'Capacidad menor o igual a 0 kg'),
    ('tipo_camion', 'FALTANTE', 'Tipo de camión vacío'),
    ('tipo_camion', 'TIPO_NO_RECONOCIDO', 'Valor distinto de Refrigerado o Seco'),
], columns=['columna_afectada', 'codigo_error', 'criterio'])

display(CATALOGO_PROBLEMAS)

def registrar(df, mascara, columna, codigo):
    mascara = mascara.fillna(False).astype(bool)
    r = df.loc[mascara, ['fila_bronze']].copy()
    r['columna_afectada'] = columna
    r['codigo_error'] = codigo
    r['valor_original'] = df.loc[mascara, columna].astype('string').to_numpy()
    return r

def diagnosticar(df):
    p = df.copy(deep=True)
    p.insert(0, 'fila_bronze', range(1, len(p) + 1))
    hallazgos = []

    # camion_id
    cid = p['camion_id'].astype('string')
    cid_limpio = cid.str.strip()

    hallazgos.append(
        registrar(p, cid_limpio.eq(''), 'camion_id', 'FALTANTE')
    )
    hallazgos.append(
        registrar(
            p,
            cid.ne(cid_limpio) & cid.ne(''),
            'camion_id',
            'FORMATO_INCONSISTENTE'
        )
    )
    hallazgos.append(
        registrar(
            p,
            ~cid_limpio.str.fullmatch(r'CAM-\d+', na=False) & cid_limpio.ne(''),
            'camion_id',
            'FORMATO_INCONSISTENTE'
        )
    )
    hallazgos.append(
        registrar(
            p,
            cid_limpio.duplicated(keep='first') & cid_limpio.ne(''),
            'camion_id',
            'DUPLICADO'
        )
    )

    # centro
    centro = p['centro_distribucion_base'].astype('string').str.strip()
    hallazgos.append(
        registrar(
            p,
            centro.eq(''),
            'centro_distribucion_base',
            'FALTANTE'
        )
    )

    # capacidad
    cap_txt = p['capacidad_kg'].astype('string').str.strip()
    cap_num = pd.to_numeric(cap_txt, errors='coerce')

    hallazgos.append(
        registrar(p, cap_txt.eq(''), 'capacidad_kg', 'FALTANTE')
    )
    hallazgos.append(
        registrar(
            p,
            cap_txt.ne('') & cap_num.isna(),
            'capacidad_kg',
            'NO_NUMERICA'
        )
    )
    hallazgos.append(
        registrar(
            p,
            cap_num.notna() & (cap_num <= 0),
            'capacidad_kg',
            'FUERA_RANGO'
        )
    )

    # tipo
    tipo = p['tipo_camion'].astype('string').str.strip()
    tipos_validos = {'Refrigerado', 'Seco'}

    hallazgos.append(
        registrar(p, tipo.eq(''), 'tipo_camion', 'FALTANTE')
    )
    hallazgos.append(
        registrar(
            p,
            tipo.ne('') & ~tipo.isin(tipos_validos),
            'tipo_camion',
            'TIPO_NO_RECONOCIDO'
        )
    )

    problemas = pd.concat(hallazgos, ignore_index=True)
    problemas['version_diagnostico'] = VERSION_DIAGNOSTICO
    problemas = problemas.sort_values(
        ['fila_bronze', 'columna_afectada', 'codigo_error'],
        kind='stable'
    ).reset_index(drop=True)

    if len(problemas):
        por_fila = problemas.groupby('fila_bronze')['columna_afectada'].agg(
            lambda x: '|'.join(dict.fromkeys(x))
        )
    else:
        por_fila = pd.Series(dtype='string')

    p['columnas_con_problemas'] = p['fila_bronze'].map(por_fila).fillna('')
    p['en_cuarentena'] = p['columnas_con_problemas'].ne('')

    return p, problemas

df_diagnosticado, df_problemas = diagnosticar(df_bronze)
df_cuarentena = df_diagnosticado[
    df_diagnosticado['en_cuarentena']
].copy()

print(
    f'Filas: {len(df_diagnosticado):,} | '
    f'Problemas: {len(df_problemas):,} | '
    f'Cuarentena: {len(df_cuarentena):,}'
)

display(
    df_problemas.groupby(
        ['columna_afectada', 'codigo_error']
    ).size().rename('conteo').reset_index()
)


,columna_afectada,codigo_error,criterio
0,camion_id,FALTANTE,Identificador vacío
1,camion_id,FORMATO_INCONSISTENTE,El identificador no sigue el patrón CAM-##
2,camion_id,DUPLICADO,Identificador repetido; se marca la aparición ...
3,centro_distribucion_base,FALTANTE,Centro de distribución vacío
4,capacidad_kg,FALTANTE,Capacidad vacía
5,capacidad_kg,NO_NUMERICA,Valor no convertible a número
6,capacidad_kg,FUERA_RANGO,Capacidad menor o igual a 0 kg
7,tipo_camion,FALTANTE,Tipo de camión vacío
8,tipo_camion,TIPO_NO_RECONOCIDO,Valor distinto de Refrigerado o Seco


Filas: 32 | Problemas: 6 | Cuarentena: 4


,columna_afectada,codigo_error,conteo
0,camion_id,DUPLICADO,2
1,camion_id,FORMATO_INCONSISTENTE,4


## 4 · Reporte y comprobaciones

In [5]:
def construir_reporte():
    conteos = df_problemas.groupby(
        ['columna_afectada', 'codigo_error']
    ).size()

    filas = [
        ('archivo_bronze', RUTA_BRONZE.name),
        ('sha256_bronze', HASH_BRONZE),
        ('version_diagnostico', VERSION_DIAGNOSTICO),
        ('filas_bronze', len(df_bronze)),
        ('filas_diagnosticadas', len(df_diagnosticado)),
        ('filas_en_cuarentena', int(df_diagnosticado['en_cuarentena'].sum())),
        ('filas_sin_cuarentena', int((~df_diagnosticado['en_cuarentena']).sum())),
        ('problemas_detectados', len(df_problemas)),
    ]

    filas += [
        (f'{col}:{codigo}', int(total))
        for (col, codigo), total in conteos.items()
    ]

    return pd.DataFrame(filas, columns=['metrica', 'valor'])

reporte_calidad = construir_reporte()

assert list(df_diagnosticado.columns) == [
    'fila_bronze',
    *COLUMNAS_ORIGINALES,
    'columnas_con_problemas',
    'en_cuarentena'
]

pd.testing.assert_frame_equal(
    df_diagnosticado[COLUMNAS_ORIGINALES],
    df_bronze[COLUMNAS_ORIGINALES]
)

assert len(df_diagnosticado) == len(df_bronze)
assert len(df_cuarentena) == int(
    df_diagnosticado['en_cuarentena'].sum()
)

display(reporte_calidad)
print('Comprobaciones previas a la exportación: correctas')


,metrica,valor
0,archivo_bronze,andinalog_flota.csv
1,sha256_bronze,5c02eb8ef591c7f98846b8f547f41383467b41e7768bdd...
2,version_diagnostico,GIAD-M3-S4-FLOTA-diagnostico-v1
3,filas_bronze,32
4,filas_diagnosticadas,32
5,filas_en_cuarentena,4
6,filas_sin_cuarentena,28
7,problemas_detectados,6
8,camion_id:DUPLICADO,2
9,camion_id:FORMATO_INCONSISTENTE,4


Comprobaciones previas a la exportación: correctas


## 5 · Exportación reproducible

Las salidas se guardan en:

`Giad/S4/andinalog_flota/notebook1/salidas/`

El origen Bronze no se modifica.


In [6]:
def exportar_salidas(tablas):
    DIRECTORIO_SALIDAS.mkdir(parents=True, exist_ok=True)
    temporales = {}

    try:
        for nombre, tabla in tablas.items():
            destino = DIRECTORIO_SALIDAS / nombre

            with tempfile.NamedTemporaryFile(
                mode='w',
                suffix='.csv',
                prefix='.tmp_flota_',
                dir=DIRECTORIO_SALIDAS,
                encoding='utf-8-sig',
                newline='',
                delete=False
            ) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)

        for destino, temporal in temporales.items():
            os.replace(temporal, destino)

    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)

    return list(temporales.keys())

tablas_salida = {
    'andinalog_flota_diagnosticado.csv': df_diagnosticado,
    'andinalog_flota_problemas.csv': df_problemas,
    'andinalog_flota_cuarentena.csv': df_cuarentena,
    'andinalog_flota_reporte_calidad.csv': reporte_calidad,
}

rutas_creadas = exportar_salidas(tablas_salida)

for ruta in rutas_creadas:
    print(ruta)

print('Bronze intacta; salidas anteriores reemplazadas')


c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_flota\notebook1\salidas\andinalog_flota_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_flota\notebook1\salidas\andinalog_flota_problemas.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_flota\notebook1\salidas\andinalog_flota_cuarentena.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_flota\notebook1\salidas\andinalog_flota_reporte_calidad.csv
Bronze intacta; salidas anteriores reemplazadas


## Siguiente etapa

El Notebook 2 leerá `andinalog_flota_diagnosticado.csv` y `andinalog_flota_problemas.csv`.

Las correcciones no se ejecutan en este Notebook 1: primero deben definirse y validarse las reglas de tratamiento.
